In [ ]:
import os
os.chdir(os.environ.get("PROJECT_ROOT", "../.."))

import pickle
from pathlib import Path
from typing import List, Tuple
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

base = Path("results/router-saturation")
assert base.is_dir(), f"Results directory not found: {base}. Run router_saturation-290m.sh first."

In [ ]:
def human_format(x, pos):
    return f"{x * 100:.0f}%"

def plot(glob: List[List[Path]], nrows: int, ncols: int, save: Path):
    sns.set_theme(style="whitegrid")
    fig, axs = plt.subplots(nrows, ncols, figsize=(16, 5), sharey=True)
    axs = axs.flatten()

    packs: List[Tuple[Path, List[float]]] = []
    steps = np.arange(0.1, 1.01, 0.1)
    steps = np.delete(steps, 2)
    for files in glob:
        items = []
        files = sorted(files, key=lambda x: int(x.parts[-2]))
        for file in files:
            with file.open("rb") as f:
                record = np.array(pickle.load(f))
                record = np.delete(record, -2) if len(record) > 10 else record
                record = np.delete(record, 2)
                items.append((file, record))
        packs.append(items)

    for ax, items in zip(axs, packs):
        for file, record in items:
            ax.plot(steps, record, marker='o', label=f"L{file.parts[-2]}")

        scale, topk = file.parts[2], file.parts[4]
        scale = scale.replace("flame-moe", "FLAME-MoE")
        scale = scale.replace("1.7b", "1.7B-10.3B")
        scale = scale.replace("721m", "721M-3.8B")
        scale = scale.replace("290m", "290M-1.3B")
        topk = topk.removesuffix(".pkl")
        ax.set_title(f"{scale}, top-{topk}", fontsize=12)
        ax.set_xlabel("Training Stage", fontsize=12)
        if ax == axs[0]: 
            ax.set_ylabel("Router Saturation", fontsize=12)

        ax.tick_params(axis='both', labelsize=12)
        ax.xaxis.set_major_formatter(FuncFormatter(human_format))
        ax.yaxis.set_major_formatter(FuncFormatter(human_format))
        ax.set_ylim(0, 1.02)
        ax.grid(False)

        if ax == axs[-1]:
            ax.legend(loc='lower right', fontsize=10, frameon=False, ncol=3)

    plt.tight_layout()
    save.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save, bbox_inches='tight', pad_inches=0.1)
    plt.close()

In [ ]:
print("router saturation, across scales, under top-1")
glob = list(map(list, [base.glob("*290m/*/1.pkl"), base.glob("*721m/*/1.pkl"), base.glob("*1.7b/*/1.pkl")]))
plot(glob, 1, 3, Path("figures/cross-scale-router-saturation/top-1.pdf"))

print("router saturation, across scales, under top-6")
glob = list(map(list, [base.glob("*290m/*/6.pkl"), base.glob("*721m/*/6.pkl"), base.glob("*1.7b/*/6.pkl")]))
plot(glob, 1, 3, Path("figures/cross-scale-router-saturation/top-6.pdf"))


In [ ]:
# Configuration for model comparison
models = [
    "290m-seed3407-32e",
    "290m-dro-activation-beta0.999-eta0.001-alpha1.0-norml2_fine_grained_32e"
]
model_labels = ["Baseline", "DRMoET"]
layers = [2, 3, 4, 5, 6, 7, 8, 9]
topks = [6, 4, 2, 1]

# Checkpoints: evenly spaced across training
checkpoints = [540, 2160, 3780, 5400, 7020, 8640, 10260, 12420, 14040, 16000]
steps = np.array(checkpoints) / 16000  # normalize to 0-1

# Create output directory
output_dir = Path("figures/router-saturation-32e")
output_dir.mkdir(parents=True, exist_ok=True)

def plot_model_comparison(models, model_labels, layers, topk, save):
    """Plot both models side-by-side for a given top-k, showing all layers."""
    sns.set_theme(style="whitegrid")
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    
    for ax, model, label in zip(axs, models, model_labels):
        for layer in layers:
            pkl_path = base / model / str(layer) / f"{topk}.pkl"
            if pkl_path.exists():
                with pkl_path.open("rb") as f:
                    record = np.array(pickle.load(f))
                    ax.plot(steps, record, marker='o', label=f"L{layer}")
        
        ax.set_title(f"{label}, top-{topk}", fontsize=14)
        ax.set_xlabel("Training Progress", fontsize=12)
        if ax == axs[0]:
            ax.set_ylabel("Router Saturation", fontsize=12)
        ax.tick_params(axis='both', labelsize=11)
        ax.xaxis.set_major_formatter(FuncFormatter(human_format))
        ax.yaxis.set_major_formatter(FuncFormatter(human_format))
        ax.set_ylim(0, 1.02)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='lower right', fontsize=9, frameon=False, ncol=2)
    
    plt.tight_layout()
    save.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save, bbox_inches='tight', pad_inches=0.1)
    plt.close()
    print(f"Saved: {save}")

def plot_direct_comparison(models, model_labels, layer, topk, save):
    """Plot both models on same axes for direct comparison."""
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    
    colors = ['#1f77b4', '#ff7f0e']  # blue, orange
    markers = ['o', 's']
    
    for model, label, color, marker in zip(models, model_labels, colors, markers):
        pkl_path = base / model / str(layer) / f"{topk}.pkl"
        if pkl_path.exists():
            with pkl_path.open("rb") as f:
                record = np.array(pickle.load(f))
                ax.plot(steps, record, marker=marker, color=color, label=label, linewidth=2, markersize=6)
    
    ax.set_title(f"Layer {layer}, top-{topk}", fontsize=14)
    ax.set_xlabel("Training Progress", fontsize=12)
    ax.set_ylabel("Router Saturation", fontsize=12)
    ax.tick_params(axis='both', labelsize=11)
    ax.xaxis.set_major_formatter(FuncFormatter(human_format))
    ax.yaxis.set_major_formatter(FuncFormatter(human_format))
    ax.set_ylim(0, 1.02)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=11, frameon=True)
    
    plt.tight_layout()
    save.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save, bbox_inches='tight', pad_inches=0.1)
    plt.close()

def plot_all_layers_comparison(models, model_labels, topk, save):
    """Plot all layers for both models in a 2x4 grid."""
    sns.set_theme(style="whitegrid")
    fig, axs = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
    axs = axs.flatten()
    
    colors = ['#1f77b4', '#ff7f0e']
    markers = ['o', 's']
    
    for ax, layer in zip(axs, layers):
        for model, label, color, marker in zip(models, model_labels, colors, markers):
            pkl_path = base / model / str(layer) / f"{topk}.pkl"
            if pkl_path.exists():
                with pkl_path.open("rb") as f:
                    record = np.array(pickle.load(f))
                    ax.plot(steps, record, marker=marker, color=color, label=label, linewidth=1.5, markersize=4)
        
        ax.set_title(f"Layer {layer}", fontsize=12)
        ax.tick_params(axis='both', labelsize=10)
        ax.xaxis.set_major_formatter(FuncFormatter(human_format))
        ax.yaxis.set_major_formatter(FuncFormatter(human_format))
        ax.set_ylim(0, 1.02)
        ax.grid(True, alpha=0.3)
    
    # Add legend to last subplot
    axs[-1].legend(loc='lower right', fontsize=9, frameon=True)
    
    # Common labels
    fig.text(0.5, 0.02, 'Training Progress', ha='center', fontsize=12)
    fig.text(0.02, 0.5, 'Router Saturation', va='center', rotation='vertical', fontsize=12)
    fig.suptitle(f'Router Saturation Comparison (top-{topk})', fontsize=14, y=0.98)
    
    plt.tight_layout(rect=[0.03, 0.03, 1, 0.96])
    save.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save, bbox_inches='tight', pad_inches=0.1)
    plt.close()
    print(f"Saved: {save}")

# Plot 1: Side-by-side comparison for each top-k
print("=== Generating side-by-side comparisons ===")
for topk in topks:
    plot_model_comparison(models, model_labels, layers, topk, output_dir / f"side-by-side-topk-{topk}.pdf")

# Plot 2: All layers in one figure (2x4 grid) for each top-k
print("\n=== Generating all-layers grid ===")
for topk in topks:
    plot_all_layers_comparison(models, model_labels, topk, output_dir / f"all-layers-topk-{topk}.pdf")

# Plot 3: Direct comparison for key layers and top-k values
print("\n=== Generating direct comparisons ===")
for layer in [2, 5, 9]:  # early, mid, late
    for topk in [6, 1]:  # full routing vs top-1
        plot_direct_comparison(models, model_labels, layer, topk, output_dir / f"direct-L{layer}-topk{topk}.pdf")

print(f"\n=== Done! Output saved to: {output_dir} ===")